In [31]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from typing import TypedDict, List, Dict
import pandas as pd
from operator import add #ajouter à chaque fois un nouveau état
from typing_extensions import Annotated, TypedDict
from qdrant_client.models import VectorParams, Distance
from langchain.tools import tool
from langchain_qdrant import QdrantVectorStore
from langchain_ollama import ChatOllama
from dotenv import  load_dotenv
from qdrant_client.models import (
    Filter,
    FieldCondition,
    GeoRadius,
    GeoPoint
    )
from qdrant_client import QdrantClient
from langchain_community.embeddings import HuggingFaceEmbeddings

#####   1 -Prétraitement et Construction de la source de données.

In [ ]:

#loading environment variables, OpenAI key and LangSmith
load_dotenv()


#model creation
model = ChatOllama(
    model="llama3.2", 
    temperature=0
)

In [33]:
df = pd.read_csv(
    "datatourisme-tour.csv", 
    sep=",", 
    encoding="utf-8"
)

In [ ]:
#location details are essential; lines without locations details must be removed.
df = df[df["Latitude"].notna()]
df = df[df["Longitude"].notna()]


# Also check for outliers:
# Latitude: 41°N → 51.5°N Longitude: -5° → 10°
# For metropolitan France.
df = df[
(df["Latitude"] > 41) &
(df["Latitude"] < 52)
] 

df = df[
(df["Longitude"] > -5) &
(df["Longitude"] < 11)
] 

# removes identical lines
df = df.drop_duplicates() 
df = df.drop_duplicates(subset=["Nom_du_POI", "Latitude", "Longitude"])


#remove unusable rows; a POI without a name is of no use.
df = df.dropna(
    subset=["Nom_du_POI"]
)

# replace missing values
df["Description"] = df["Description"].fillna("")

In [ ]:
#Document creation
def build_document(row) :
    return f"""
Nom_du_POI : {row['Nom_du_POI']}
Categories_de_POI : {row['Categories_de_POI']}
Adresse_postale : {row['Adresse_postale']}
Code_postal_et_commune: {row['Code_postal_et_commune']}
Description: {row['Description']}
""".strip()

df["document"] = df.apply(
    build_document,
    axis=1
)

In [68]:
# Setting up metadata
def build_metadata(row):
    metadata = {
        "Nom_du_POI" : row['Nom_du_POI'], 
        "Categories_de_POI" : row['Categories_de_POI'], 
        "location": {
            "lat": row["Latitude"],
            "lon": row["Longitude"]
        },
        "Code_postal_et_commune" : row['Code_postal_et_commune']
    }
    return metadata

df["metadata"] = df.apply(
    build_metadata,
    axis=1
)

In [69]:
documents = df["document"].tolist()
metadatas = df["metadata"].tolist()

# Generation of documents and metadata for indexing
from langchain_core.documents import Document
docs_metadata = [
    Document(
        page_content=doc,
        metadata=meta
    )
    for doc, meta in zip(df["document"], df["metadata"])
]

##### 2- vectorisation. 

In [70]:

#Initializing the connection to the vector store server
client = QdrantClient(
    url="http://localhost:6333"
)

#
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


client.recreate_collection(
    collection_name="tourisme_tour_france",
    vectors_config=VectorParams(
        size=384,  # IMPORTANT: depends on your embedding template
        distance=Distance.COSINE
    )
)

#creation of a persistent vector_store
v_stores = QdrantVectorStore(
        client=client,
        collection_name="tourisme_tour_france",
        embedding=embeddings
    )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5650.33it/s]
/var/folders/kw/zr1physd0vq7109nxm533f4h0000gn/T/ipykernel_86289/1411758070.py:10: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


In [71]:

#initialization and creation of chunks for documents and metadata
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200, 
    add_start_index=True
)
all_splits = text_splitter.split_documents(docs_metadata)
print(len(all_splits))

27543


In [72]:
from qdrant_client.models import Distance, VectorParams

collections = client.get_collections().collections
existing = [c.name for c in collections]

if "tourisme_tour_france" not in existing:
    client.recreate_collection    (
        collection_name="tourisme_tour_france",
        vectors_config=VectorParams(
            size=384,
            distance=Distance.COSINE
        )
    )
v_stores.add_documents(all_splits)

['acf2fdf23dac4d2aa7f4784c41957a96',
 'bb9a3ef4733947e4b90b198e0689e6f7',
 'e40687374b61420b94d4fcb125d4ff6a',
 'd4f353247094434cbbcfc9f6621bb9f1',
 'c2252df95cae48648d61416754b6b2df',
 '3382c3a78e8749a68858662f761461cb',
 'd17801a7206c49e5a669fd2a67375f3c',
 '0f247013463b4b24ba0007d10ed21bfc',
 '85f702384bef4fb18abdfe3ac0dd1980',
 '069f86ebc7544d8590d6b295bf5ebbdd',
 'f45cdf4c63b642988d82cb41eb273552',
 'e28153a6716942bfb71a38ca37195ca9',
 'ead0a3b55b6645678be049729c8c7060',
 '07ec32f4dac14f4d86fec747d365a61d',
 '326714cb300548598d43f99be03f7c0f',
 'b5ec6a1b778d4b4aa21b7a8acc8c32be',
 'f071986d59aa46fd902774eb11eba0ad',
 'da852b7f715c4b36a96e78d5e2deb804',
 '4031b89af1bc46a895ff9311d2335094',
 '21dbc2d2711d42c8bdea5fc4d8af75b6',
 'ef014deaa6344819b8ad2b8ad436f21e',
 '842dd9b9b29b47a5af4f6070bfb763e3',
 'daeb741b5c644e8289b3559c9fd7c273',
 '0b419e26c0d14d13a3c1df91fdb94c95',
 '87ad4fdcdb2f420f8bca099daf60c486',
 '145ac6bb62df40919a2f811fe2f6c871',
 'df13c468a1f64b69bc66c5a0d3e1d339',
 

##### 3- Développement des outils 

In [91]:
# Tool to search for POI - Tool 1 : Qdrant vector search
@tool
def search_poi(query:str ) : 
    """
    Recherche des points d'intérêt touristiques en France à l'aide d'une recherche vectorielle dans la base Qdrant.
    À utiliser lorsque l'utilisateur demande des monuments, musées, châteaux, plages, parcs ou tout autre lieu touristique.
    """
    docs = v_stores.similarity_search(query, k=5)

    return[
        {
            "Nom_du_POI": d.metadata.get("Nom_du_POI"),
            "Categories_de_POI": d.metadata.get("Categories_de_POI"),
            "Adresse_postale": d.metadata.get("Adresse_postale"), 
            "Code_postal_et_commune": d.metadata.get("Code_postal_et_commune"), 
            "Description": d.metadata.get("Description") #d.page_content
        }
        for d in docs
    ] 

@tool
def search_nearby(latitude: float, longitude: float, radius_km: float = 10, limit: int = 10):
    """
   
   Recherche des points d'intérêt touristiques en France situés à proximité d'une position géographique donnée.
   À utiliser lorsque l'utilisateur demande des lieux proches d'une ville, d'une adresse ou d'une position. 

    """
    points, _ = client.scroll(
        collection_name="tourisme_tour_france",
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="location",
                    geo_radius=GeoRadius(
                        center=GeoPoint(
                            lat=latitude,
                            lon=longitude
                        ),
                        radius=radius_km * 1000  # mètres
                    )
                )
            ]
        ),
        limit=limit,
        with_payload=True
    )

    return [
          {
            "Nom_du_POI": p.payload.get("Nom_du_POI"),
            "Categories_de_POI": p.payload.get("Categories_de_POI"),
            "Adresse_postale" : p.payload.get("Adresse_postale"), 
            "Code_postal_et_commune": p.payload.get("Code_postal_et_commune"),
            "Latitude": p.payload.get("Latitude"),
            "Longitude": p.payload.get("Longitude"),
            "Description": p.payload.get("Description")
        }
        for p in points
    ]

@tool
def format_poi (pois) : 
    """formatage des POIs"""
    return "\n".join([
        f"{p['Nom_du_POI']} - {p['Code_postal_et_commune']} ({p['Description']})"
        for p in pois
    ])

tools = [search_poi, search_nearby, format_poi]
#on va utiliser uen fonctions bindtool pour lier le, modèle avec les toolsq qu'il peut utiliser
tools_by_name = {tool.name: tool for tool in tools}
#on passe au modèle une liste des tools qui seront utilisés par ce dernier
model_with_tools = model.bind_tools(tools)

##### 4- Développement de l’architecture du graphe LangGraph – mise en œuvre du RAG Agentic. 

##### 4.1-défnition du State

In [92]:
#State is important. We keep track of all previous  responses
class State(TypedDict):
    query: str
    messages : Annotated[list, add]
    pois : list[Dict]
    answer: str
    steps : int

##### 4.2-Noeud Agent

In [93]:
# At this node of the graph, the agent maintains the conversational context, 
# which is enriched during subsequent calls, with the aim of producing a 
# more precise response.
from langchain_core.messages import HumanMessage, SystemMessage
def agent(state: State):
    if not state.get("messages") : # LLM subsequent call if required
        messages = [
        SystemMessage(content=
            """
            Tu es un assistant touristique spécialisé sur la France Métropolitaine
               
            """
        ),
        HumanMessage(content=state["query"])
    ] 
    else: 
        messages = state["messages"] #first call only

    # LLM request tools the use of tools (response will containt ToolMessage)
    response = model_with_tools.invoke(messages)

    return {
        "messages": state.get("messages", []) + [response], #System, Human, AiMessage, tool_call -> ToolMessage 
        "steps": state.get("steps", 0) + 1
    }

##### 4.3-  Noeud de Tools

In [94]:
from langgraph.prebuilt import ToolNode
# if needed, LLM request to use all existing tools to answser the question
tool_node = ToolNode(tools) # ToolMessage

##### 4.4 -Fonction de routage

In [95]:
#What next ? tool_call or end the program ? 
def should_continue(state):
    if state.get("steps", 0) > 3:
        return "end"
    last_message = state["messages"][-1]
    #print(type(last_message))
    #print(last_message)

    # decide whether there is a need to call a tool
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"

    # if not end the program and give hand back to LLM
    return "end"

##### 4.5- Noeud de fin

In [96]:
# we are moving to the end
def finalize(state: State):
      # chercher le dernier message du LLM (pas tool)
    for msg in reversed(state["messages"]):
        if hasattr(msg, "content") and msg.content:
            return {"answer": msg.content}

    return {"answer": "Aucune réponse générée"}

#### 5. Orchestration LangGraph

In [97]:
from langgraph.graph import StateGraph, START,END

graph = StateGraph(State)

graph.add_node("agent", agent)
graph.add_node("tools", tool_node)
graph.add_node("final", finalize)

graph.add_edge(START, "agent")
graph.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        "end": "final"
    }
)
graph.add_edge("tools", "agent")
graph.add_edge("final", END)

app = graph.compile()

#### 6. Exécution

In [106]:
result = app.invoke({
    "query": "Je dispose d’un budget de 300€ pour un week-end en France : quelle destination touristique me recommandes-tu et pourquoi ?",
    "messages": [],
    "pois": []
})
#print(result)
print(result["answer"])

Based on the search results, here are some nearby locations:

1. The Eiffel Tower (Paris, France) - 0.5 km away
2. The Louvre Museum (Paris, France) - 0.7 km away
3. Notre-Dame Cathedral (Paris, France) - 0.8 km away
4. Arc de Triomphe (Paris, France) - 1.1 km away
5. Champs-Élysées (Paris, France) - 1.2 km away

Would you like more information about any of these locations?


In [ ]:
#df.info()
#Conserver uniquement ce qui apporte du contexte touristique.
#print(df.isnull().sum()) # nombre de valeur manquante
#df.head()
#print(df['Description'].isnull().sum())
#print(df['Description'].isnull().sum())